In [1]:
# =============================================================================
# Cell 1 - bootstrap. RESUMABLE: if the prepared working frame already exists on
# Drive, the download and thinning are skipped entirely. A Colab timeout costs
# nothing once cell 4 has run once.
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, glob, subprocess, hashlib
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
for m in ['config','conformal']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config, conformal
import numpy as np, pandas as pd
for d in (config.PROC_DIR, config.REPORTS_DIR, config.INTERIM_DIR):
    d.mkdir(parents=True, exist_ok=True)
ALPHA=config.ALPHA_PRIMARY
RAW_DIR   = Path('/content/ciciot2023_raw')          # session-local, never persisted
PREPARED  = config.PROC_DIR/'ciciot2023_prepared.parquet'   # the artefact that matters
PER_LABEL_CAP = 60000
NULL_TOKENS = {'nan','none','null','','na'}
RESUME = PREPARED.exists()
print('prepared frame already on Drive:', RESUME, '|', PREPARED)
if RESUME:
    print('  -> cells 2 to 4 will skip the download and reuse it')
st=os.statvfs('/content/drive/MyDrive'); print(f'Drive free: {st.f_bavail*st.f_frsize/1e9:.1f} GB')


Mounted at /content/drive
prepared frame already on Drive: False | /content/drive/MyDrive/CALSHIFT_Research/calshift-research/data/processed/ciciot2023_prepared.parquet
Drive free: 210.0 GB


In [5]:
from google.colab import files, drive
from pathlib import Path
import os, shutil

drive.mount('/content/drive', force_remount=False)
print('Select the kaggle.json you just downloaded:')
up = files.upload()                      # pick kaggle.json in the file dialog
assert 'kaggle.json' in up, 'the file must be named kaggle.json'

os.makedirs('/root/.kaggle', exist_ok=True)
Path('/root/.kaggle/kaggle.json').write_bytes(up['kaggle.json'])
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# persist to Drive so notebook 31 finds it automatically next time
dst = Path('/content/drive/MyDrive/kaggle.json')
shutil.copy('/root/.kaggle/kaggle.json', dst); os.chmod(dst, 0o600)
print('installed at /root/.kaggle/kaggle.json and saved to', dst)

# verify the credential actually works before committing to a 13 GB download
import subprocess
subprocess.run(['pip','install','-q','kaggle'], capture_output=True)
r = subprocess.run(['kaggle','datasets','list','-s','unb-cic-iot'], capture_output=True, text=True)
print((r.stdout + r.stderr)[:600])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Select the kaggle.json you just downloaded:


Saving kaggle.json to kaggle.json
installed at /root/.kaggle/kaggle.json and saved to /content/drive/MyDrive/kaggle.json
ref                                                              title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
---------------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
madhavmalhotra/unb-cic-iot-dataset                          


In [6]:
# =============================================================================
# Cell 2 - DOWNLOAD to session-local disk (not Drive). The raw release is ~13 GB
# and is disposable: only the thinned working frame is persisted. Skipped on resume.
# =============================================================================
KAGGLE_DATASET = 'madhavmalhotra/unb-cic-iot-dataset'
csvs = []
if RESUME:
    print('resume: skipping download')
else:
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    KJ = DRIVE_ROOT/'kaggle.json'
    if KJ.exists():
        os.makedirs('/root/.kaggle', exist_ok=True)
        shutil.copy(KJ, '/root/.kaggle/kaggle.json'); os.chmod('/root/.kaggle/kaggle.json', 0o600)
        print('kaggle credentials restored from Drive')
    else:
        raise SystemExit(f'place kaggle.json at {KJ} (kaggle.com/settings -> Create New Token)')
    subprocess.run(['pip','install','-q','kaggle'], capture_output=True, text=True)
    r = subprocess.run(['kaggle','datasets','download','-d',KAGGLE_DATASET,'-p',str(RAW_DIR),'--unzip'],
                       capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1500:])
    csvs = sorted(glob.glob(str(RAW_DIR/'**'/'*.csv'), recursive=True))
    print(f'\nCSV files: {len(csvs)} | {sum(Path(f).stat().st_size for f in csvs)/1e9:.2f} GB')
    if not csvs:
        raise SystemExit('download produced no CSVs; check the Kaggle dataset slug and token')


kaggle credentials restored from Drive
[00:28<00:02, 135MB/s]
 89%|████████▊ | 2.46G/2.77G [00:28<00:02, 132MB/s]
 89%|████████▉ | 2.47G/2.77G [00:28<00:02, 133MB/s]
 90%|████████▉ | 2.49G/2.77G [00:28<00:02, 131MB/s]
 90%|█████████ | 2.50G/2.77G [00:28<00:02, 120MB/s]
 90%|█████████ | 2.51G/2.77G [00:29<00:02, 115MB/s]
 91%|█████████ | 2.52G/2.77G [00:29<00:02, 118MB/s]
 91%|█████████▏| 2.53G/2.77G [00:29<00:02, 122MB/s]
 92%|█████████▏| 2.55G/2.77G [00:32<00:19, 12.4MB/s]
 92%|█████████▏| 2.55G/2.77G [00:32<00:15, 15.1MB/s]
 93%|█████████▎| 2.57G/2.77G [00:32<00:08, 24.9MB/s]
 93%|█████████▎| 2.59G/2.77G [00:32<00:06, 30.9MB/s]
 94%|█████████▎| 2.60G/2.77G [00:33<00:05, 34.0MB/s]
 94%|█████████▍| 2.61G/2.77G [00:33<00:04, 37.0MB/s]
 94%|█████████▍| 2.61G/2.77G [00:33<00:04, 39.9MB/s]
 95%|█████████▍| 2.63G/2.77G [00:33<00:03, 49.6MB/s]
 95%|█████████▌| 2.64G/2.77G [00:33<00:02, 56.9MB/s]
 95%|█████████▌| 2.65G/2.77G [00:33<00:01, 68.2MB/s]
 96%|█████████▌| 2.66G/2.77G [00:33<00:01, 7

In [7]:
# =============================================================================
# Cell 3 - INVENTORY labels (cheap: label column only). Skipped on resume.
# =============================================================================
if RESUME:
    print('resume: skipping inventory')
    LABEL_COL = 'label'
else:
    head = pd.read_csv(csvs[0], nrows=5)
    LABEL_COL = next((c for c in head.columns if c.strip().lower() in ('label','labels','attack','class')), None)
    assert LABEL_COL, f'no label column in {list(head.columns)}'
    print(f'columns: {len(head.columns)} | label column: {LABEL_COL}')
    from collections import Counter
    cnt = Counter()
    for i,f in enumerate(csvs):
        cnt.update(pd.read_csv(f, usecols=[LABEL_COL])[LABEL_COL].astype(str).str.strip()
                     .value_counts().to_dict())
        if (i+1) % 40 == 0: print(f'  scanned {i+1}/{len(csvs)}')
    inv = pd.Series(cnt).sort_values(ascending=False)
    nulls = [l for l in inv.index if str(l).strip().lower() in NULL_TOKENS]
    if nulls:
        print(f'dropping null label token(s) {nulls} covering {int(inv[nulls].sum())} row(s)')
        inv = inv.drop(index=nulls)
    print(f'\ntotal rows: {inv.sum():,} | labels: {len(inv)}')
    print(inv.to_string())


columns: 47 | label column: label
  scanned 40/169
  scanned 80/169
  scanned 120/169
  scanned 160/169

total rows: 46,686,579 | labels: 34
DDoS-ICMP_Flood            7200504
DDoS-UDP_Flood             5412287
DDoS-TCP_Flood             4497667
DDoS-PSHACK_Flood          4094755
DDoS-SYN_Flood             4059190
DDoS-RSTFINFlood           4045285
DDoS-SynonymousIP_Flood    3598138
DoS-UDP_Flood              3318595
DoS-TCP_Flood              2671445
DoS-SYN_Flood              2028834
BenignTraffic              1098195
Mirai-greeth_flood          991866
Mirai-udpplain              890576
Mirai-greip_flood           751682
DDoS-ICMP_Fragmentation     452489
MITM-ArpSpoofing            307593
DDoS-UDP_Fragmentation      286925
DDoS-ACK_Fragmentation      285104
DNS_Spoofing                178911
Recon-HostDiscovery         134378
Recon-OSScan                 98259
Recon-PortScan               82284
DoS-HTTP_Flood               71864
VulnerabilityScan            37382
DDoS-HTTP_Flood    

In [8]:
# =============================================================================
# Cell 4 - TAXONOMY, THIN, and PERSIST. This is the expensive step, so the frame
# is written to Drive immediately at the end. Everything after this cell is cheap
# and re-runnable. Skipped on resume.
# family = class (Mondrian + focal-class rule), specific attack = subtype (drives S_sup).
# =============================================================================
def to_family(lbl):
    low = str(lbl).strip().lower()
    if low.startswith('benign'):  return 'Benign'
    if low.startswith('ddos'):    return 'DDoS'
    if low.startswith('dos'):     return 'DoS'
    if low.startswith('mirai'):   return 'Mirai'
    if low.startswith('recon') or 'scan' in low or 'sweep' in low or 'discovery' in low: return 'Recon'
    if 'spoofing' in low:         return 'Spoofing'
    if 'bruteforce' in low or 'dictionary' in low: return 'BruteForce'
    return 'Web'   # residual; guarded below
def to_subtype(lbl):
    L = str(lbl).strip()
    for sep in ('-','_'):
        if sep in L:
            head, rest = L.split(sep,1)
            if head.lower() in ('ddos','dos','mirai','recon'): return rest
    return L

if RESUME:
    iot = pd.read_parquet(PREPARED)
    print('resume: loaded prepared frame', iot.shape)
else:
    KNOWN_WEB = {'BrowserHijacking','CommandInjection','SqlInjection','XSS',
                 'Uploading_Attack','Backdoor_Malware'}
    tax = pd.DataFrame({'label': inv.index, 'n': inv.values})
    tax['family']=tax['label'].map(to_family); tax['subtype']=tax['label'].map(to_subtype)
    residual = set(tax[tax.family=='Web']['label']) - KNOWN_WEB
    assert not residual, f'unexpected labels absorbed into residual Web family: {sorted(residual)}'
    print('residual-family guard: clean')
    fam=(tax.groupby('family').agg(rows=('n','sum'), n_subtypes=('subtype','nunique'))
            .sort_values('rows'))
    print('\nFAMILY INVENTORY:'); print(fam.to_string())

    keep_frac = {l: min(1.0, PER_LABEL_CAP/float(n)) for l,n in inv.items()}
    parts=[]
    for i,f in enumerate(csvs):
        d = pd.read_csv(f); d[LABEL_COL]=d[LABEL_COL].astype(str).str.strip()
        d = d[~d[LABEL_COL].str.lower().isin(NULL_TOKENS)]
        rng_f = np.random.default_rng(int(hashlib.sha256(Path(f).name.encode()).hexdigest(),16)%(2**32))
        fr = d[LABEL_COL].map(keep_frac).fillna(0.0).to_numpy(dtype=float)
        parts.append(d[rng_f.random(len(d)) < fr]); del d
        if (i+1)%40==0: print(f'  thinned {i+1}/{len(csvs)}')
    iot = pd.concat(parts, ignore_index=True).reset_index(drop=True); del parts
    iot['family']=iot[LABEL_COL].map(to_family); iot['subtype']=iot[LABEL_COL].map(to_subtype)
    assert iot['family'].notna().all()
    print('\nworking frame:', iot.shape)
    iot.to_parquet(PREPARED, index=False)      # persist NOW, before anything else can fail
    print('PERSISTED to Drive:', PREPARED, f'({PREPARED.stat().st_size/1e6:.0f} MB)')
    tax.to_csv(config.REPORTS_DIR/'label_taxonomy_ciciot2023.csv', index=False)
print('\nrows per family:'); print(iot['family'].value_counts().to_string())


residual-family guard: clean

FAMILY INVENTORY:
                rows  n_subtypes
family                          
BruteForce     13064           1
Web            24829           6
Recon         354565           5
Spoofing      486504           2
Benign       1098195           1
Mirai        2634124           3
DoS          8090738           4
DDoS        33984560          12
  thinned 40/169
  thinned 80/169
  thinned 120/169
  thinned 160/169

working frame: (1510142, 49)
PERSISTED to Drive: /content/drive/MyDrive/CALSHIFT_Research/calshift-research/data/processed/ciciot2023_prepared.parquet (134 MB)

rows per family:
family
DDoS          652205
DoS           240154
Recon         219595
Mirai         180521
Spoofing      119778
Benign         59996
Web            24829
BruteForce     13064


In [9]:
# =============================================================================
# Cell 5 - FEATURES and PARTITIONS (cheap, always runs).
# =============================================================================
DROP_PAT = ('unnamed','index','id','timestamp','time','flow_id','src_ip','dst_ip')
FEATS = [c for c in iot.columns
         if c not in (LABEL_COL,'family','subtype','partition')
         and pd.api.types.is_numeric_dtype(iot[c])
         and not (c.strip().lower().startswith('unnamed') or c.strip().lower() in DROP_PAT)]
excluded=[c for c in iot.columns if c not in FEATS and c not in (LABEL_COL,'family','subtype','partition')]
print('features kept:', len(FEATS), '| excluded:', excluded or 'none')
iot[FEATS]=iot[FEATS].replace([np.inf,-np.inf], np.nan)
iot[FEATS]=iot[FEATS].fillna(iot[FEATS].median(numeric_only=True))
const=[c for c in FEATS if iot[c].nunique(dropna=False)<=1]
if const: print('constant features dropped:', const); FEATS=[c for c in FEATS if c not in const]
print('final features:', len(FEATS))

def strat(df, fr, seed, col):
    rng=np.random.default_rng(seed); nm=list(fr); ff=np.array([fr[k] for k in nm],float)
    big=nm[int(np.argmax(ff))]; a=pd.Series(index=df.index, dtype=object)
    for _, s in df.groupby(col, sort=True):
        idx=s.index.to_numpy().copy(); rng.shuffle(idx); n=len(idx)
        c=np.floor(ff*n).astype(int); c[nm.index(big)] += n-c.sum(); k=0
        for a2,q in zip(nm,c): a.loc[idx[k:k+q]]=a2; k+=q
    return a
IOT_PARTITION_SEED = 20260726
iot['partition']=strat(iot, config.SPLIT_FRACTIONS, IOT_PARTITION_SEED, LABEL_COL).values
print('\npartitions:'); print(iot['partition'].value_counts().to_string())
assert iot['partition'].notna().all()


features kept: 46 | excluded: none
constant features dropped: ['Telnet', 'SMTP']
final features: 44

partitions:
partition
train              906138
probcal            226503
source_cal_pool    226503
val                150998


In [10]:
# =============================================================================
# Cell 6 - FEASIBILITY and FOCAL-CLASS RECORD. The gate: committed before any
# coverage number exists. Section 7.6 floor + section 12 rarest-feasible rule,
# with the added constraint that the focal class needs >=2 subtypes so a
# variant-holdout ladder can shift it while keeping it feasible under SHC.
# =============================================================================
pool = iot[iot.partition=='source_cal_pool']; need = conformal.min_calib_n(ALPHA)
rows=[]
for f,g in iot.groupby('family'):
    n=int((pool['family']==f).sum())
    rows.append({'family':f,'rows':int(len(g)),'n_source_cal_pool':n,'min_needed':need,
                 'feasible':bool(n>=need),'n_subtypes':int(g['subtype'].nunique()),
                 'ladder_capable':bool(n>=need and g['subtype'].nunique()>=2)})
feas=pd.DataFrame(rows).sort_values('n_source_cal_pool')
print(f'FEASIBILITY at alpha={ALPHA} (need {need} source calibration points):')
print(feas.to_string(index=False))

cand=feas[(feas.family!='Benign')&(feas.ladder_capable)].sort_values('n_source_cal_pool')
assert len(cand)>0, 'no attack family is both feasible and multi-subtype'
FOCAL=cand.iloc[0]['family']
print(f'\nFOCAL CLASS: {FOCAL}  (source cal pool {int(cand.iloc[0]["n_source_cal_pool"])} rows)')
print('  subtypes:', sorted(iot[iot.family==FOCAL]['subtype'].unique()))
single=feas[(feas.family!='Benign')&(feas.feasible)&(feas.n_subtypes<2)]['family'].tolist()
print('  rarer but single-subtype, so not shiftable:', single or 'none')
excl=feas[(feas.family!='Benign')&(~feas.feasible)]['family'].tolist()
print('  excluded as infeasible:', excl or 'none')

capped_share=iot['family'].value_counts(normalize=True).round(5).to_dict()
record={'dataset':'ciciot2023','alpha':ALPHA,'min_calib_needed':need,'focal_class':FOCAL,
  'focal_selection_rule':'rarest attack family satisfying section 7.6 in the source calibration '
     'pool AND holding >=2 subtypes, so a variant-holdout ladder can shift it while keeping it '
     'feasible under SHC (constraint carried from the CIC-IDS2017 experience, Amendment 9)',
  'focal_subtypes':sorted(iot[iot.family==FOCAL]['subtype'].unique().tolist()),
  'rarer_but_single_subtype':single,'excluded_infeasible':excl,
  'per_label_cap':PER_LABEL_CAP,'partition_seed':IOT_PARTITION_SEED,
  'prior_after_cap':capped_share,'n_features':len(FEATS),'label_column':LABEL_COL,
  'kaggle_mirror':'madhavmalhotra/unb-cic-iot-dataset',
  'version_of_record_note':'this mirror is not the full published release; record its file and '
     'row counts as the version of record rather than citing the canonical CIC totals',
  'feasibility_table':feas.to_dict('records'),'recorded_before_any_coverage':True}
(config.REPORTS_DIR/'focal_class_record_ciciot2023.json').write_text(json.dumps(record,indent=2,default=str))
feas.to_csv(config.REPORTS_DIR/'feasibility_binding_ciciot2023.csv',index=False)
h=hashlib.sha256(PREPARED.read_bytes()).hexdigest()
(config.REPORTS_DIR/'ciciot2023_prepared_fingerprint.json').write_text(json.dumps(
  {'path':str(PREPARED),'rows':int(len(iot)),'sha256':h,'features':FEATS},indent=2))
print('\nrecorded. frame sha256', h[:16])


FEASIBILITY at alpha=0.05 (need 19 source calibration points):
    family   rows  n_source_cal_pool  min_needed  feasible  n_subtypes  ladder_capable
BruteForce  13064               1959          19      True           1           False
       Web  24829               3720          19      True           6            True
    Benign  59996               8999          19      True           1           False
  Spoofing 119778              17966          19      True           2            True
     Mirai 180521              27077          19      True           3            True
     Recon 219595              32937          19      True           5            True
       DoS 240154              36021          19      True           4            True
      DDoS 652205              97824          19      True          12            True

FOCAL CLASS: Web  (source cal pool 3720 rows)
  subtypes: ['Backdoor_Malware', 'BrowserHijacking', 'CommandInjection', 'SqlInjection', 'Uploading_Attack'

In [11]:
# =============================================================================
# Cell 7 - commit the gate artefacts (raw data and the frame are gitignored)
# =============================================================================
def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','nb31: CIC-IoT-2023 acquire, thin, partition, feasibility and focal-class record (before any coverage)')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


[main 88a805c] nb31: CIC-IoT-2023 acquire, thin, partition, feasibility and focal-class record (before any coverage)
 7 files changed, 208 insertions(+), 2 deletions(-)
 rewrite notebooks/31_ciciot2023_gate.ipynb (98%)
 create mode 100644 notebooks/31_ciciot2023_prepare_and_gate.ipynb
 create mode 100644 reports/ciciot2023_prepared_fingerprint.json
 create mode 100644 reports/feasibility_binding_ciciot2023.csv
 create mode 100644 reports/focal_class_record_ciciot2023.json
 create mode 100644 reports/label_taxonomy_ciciot2023.csv
Branch 'main' set up to track remote branch 'main' from 'origin'.
To https://github.com/anasbiswas1/calshift-research.git
   0372432..88a805c  main -> main
88a805c nb31: CIC-IoT-2023 acquire, thin, partition, feasibility and focal-class record (before any coverage)
0372432 nb30a: CIC-IoT-2023 acquisition manifest and label inventory (no coverage computed)
9f9dd6c nb30: label-free selective prediction - risk-coverage tradeoff, escalation rate to restore nominal 

In [12]:
# =============================================================================
# Commit Amendment 10 AND correct the version-of-record note in the focal record.
# The record currently says the mirror is not the full release. It is: 169 files,
# 46,686,579 rows, 34 labels. A false disclosure is worse than none, so it is
# corrected here, before any coverage number exists.
# =============================================================================
import base64, json, subprocess, os
from pathlib import Path
os.chdir('/content/drive/MyDrive/CALSHIFT_Research/calshift-research')

Path('preregistration_amendment_10.md').write_text(base64.b64decode("IyBQcmVyZWdpc3RyYXRpb24gQW1lbmRtZW50IDEwCgoqKlN0YXR1czoqKiBiaW5kaW5nIG9uIGNvbW1pdC4gTWFkZSBhZnRlciB0aGUgQ0lDLUlvVC0yMDIzIHBhcnRpdGlvbnMsIGZlYXNpYmlsaXR5IHRhYmxlIGFuZApmb2NhbC1jbGFzcyByZWNvcmQgd2VyZSBwcm9kdWNlZCAobm90ZWJvb2sgMzEpIGJ1dCAqKmJlZm9yZSBhbnkgQ0lDLUlvVC0yMDIzIGNvdmVyYWdlIG51bWJlcgppcyBjb21wdXRlZCoqOyBgcmVwb3J0cy9gIGNvbnRhaW5zIG5vIGAqX2NpY2lvdDIwMjNgIGNvdmVyYWdlIG91dHB1dCBhdCB0aGUgdGltZSBvZiB3cml0aW5nLgoqKlNjb3BlOioqIG9wZXJhdGlvbmFsaXplcyB0aGUgQ0lDLUlvVC0yMDIzIGVudmlyb25tZW50IG5hbWVkIGluIHByZXJlZ2lzdHJhdGlvbiBzZWN0aW9ucyAzLAoxMC4xIGFuZCAxMS4xLiBDaGFuZ2VzIG5vdGhpbmcgYWJvdXQgTlNMLUtERCwgQ0lDLUlEUzIwMTcgb3IgVUdSJzE2LCB3aG9zZSByZXN1bHRzIGFyZQphbHJlYWR5IHJlY29yZGVkLgoqKk1vdGl2YXRpb246KiogQ0lDLUlvVC0yMDIzIGlzIHRoZSAibW9kZXJuIGhldGVyb2dlbmVvdXMsIGZhbWlseSBob2xkb3V0IiBlbnZpcm9ubWVudC4gSXQgd2FzCmNvbW1pdHRlZCB0byBpbiB0aGUgYmFzZSBwcmVyZWdpc3RyYXRpb24gYW5kIG5vdCBleGVjdXRlZCB1bnRpbCBub3cuIEl0cyBhbmFseXNpcyBkZWNpc2lvbnMKYXJlIHRpbWVzdGFtcGVkIGFoZWFkIG9mIGl0cyByZXN1bHRzLCBvbiB0aGUgc2FtZSBwcmluY2lwbGUgYXMgdGhlIHNlY3Rpb24gMTIgZm9jYWwtY2xhc3MgcnVsZQphbmQgQW1lbmRtZW50IDUgZm9yIENJQy1JRFMyMDE3LgoKLS0tCgojIyBBMTAuMSBWZXJzaW9uIG9mIHJlY29yZAoKVGhlIHJlbGVhc2UgdXNlZCBpcyB0aGUgZnVsbCBDSUMtSW9ULTIwMjMgQ1NWIGRpc3RyaWJ1dGlvbiBvYnRhaW5lZCB0aHJvdWdoIHRoZSBLYWdnbGUgbWlycm9yCmBtYWRoYXZtYWxob3RyYS91bmItY2ljLWlvdC1kYXRhc2V0YDogKioxNjkgQ1NWIGZpbGVzLCA0Niw2ODYsNTc5IHJvd3MsIDM0IGxhYmVscyoqLCB3aGljaAptYXRjaGVzIHRoZSBwdWJsaXNoZWQgcmVsZWFzZS4gVGhlIHByZXBhcmVkIHdvcmtpbmcgZnJhbWUgaXMgcGlubmVkIGJ5IFNIQS0yNTYgaW4KYHJlcG9ydHMvY2ljaW90MjAyM19wcmVwYXJlZF9maW5nZXJwcmludC5qc29uYC4KCkEgZmlyc3QgYWNxdWlzaXRpb24gYXR0ZW1wdCBpbiBhbiBlYXJsaWVyIHNlc3Npb24gdGVybWluYXRlZCBlYXJseSBhbmQgeWllbGRlZCAxNjIgZmlsZXMgYW5kCjQ0LDAyNSw2NjYgcm93cy4gVGhhdCBwYXJ0aWFsIGRvd25sb2FkIHdhcyAqKm5vdCoqIHVzZWQgZm9yIGFueSBhbmFseXNpczsgaXQgaXMgcmVjb3JkZWQgaGVyZQpvbmx5IHNvIHRoZSBkaXNjcmVwYW5jeSBiZXR3ZWVuIHRoZSB0d28gYWNxdWlzaXRpb24gbWFuaWZlc3RzIGluIHRoZSByZXBvc2l0b3J5IGlzIGV4cGxhaW5lZC4KCiMjIEExMC4yIENsYXNzIHRheG9ub215CgpGb2xsb3dpbmcgdGhlIEFtZW5kbWVudCA1IEE1LjIgY29udmVudGlvbjogdGhlIGJyb2FkIGF0dGFjayBmYW1pbHkgaXMgdGhlICoqY2xhc3MqKiBvbiB3aGljaApNb25kcmlhbiBjYWxpYnJhdGlvbiBhbmQgdGhlIGZvY2FsLWNsYXNzIHJ1bGUgb3BlcmF0ZTsgdGhlIHNwZWNpZmljIGF0dGFjayBpcyB0aGUgKipzdWJ0eXBlKioKdGhhdCBkcml2ZXMgYFNfc3VwYCB1bmRlciB2YXJpYW50IGhvbGRvdXQuIEJlbmlnbiBpcyB0aGUgbmVnYXRpdmUgY2xhc3MuCgpgYGAKQmVuaWduICAgICAgICAxIGxhYmVsCkREb1MgICAgICAgICAxMiBzdWJ0eXBlcyAgICAgIERvUyAgICAgICAgICAgNCBzdWJ0eXBlcwpSZWNvbiAgICAgICAgIDUgc3VidHlwZXMgICAgICBNaXJhaSAgICAgICAgIDMgc3VidHlwZXMKV2ViICAgICAgICAgICA2IHN1YnR5cGVzICAgICAgU3Bvb2ZpbmcgICAgICAyIHN1YnR5cGVzCkJydXRlRm9yY2UgICAgMSBzdWJ0eXBlCmBgYAoKTWFwcGluZyBpcyBkZXJpdmVkIGZyb20gdGhlIGxhYmVsIHN0cmluZywgYW5kIGFueSBsYWJlbCBub3QgbWF0Y2hpbmcgYW4gZXhwbGljaXQgcnVsZSBhbmQgbm90Cm9uIHRoZSBrbm93biB3ZWItYXR0YWNrIGxpc3QgaGFsdHMgdGhlIG5vdGVib29rIHJhdGhlciB0aGFuIGJlaW5nIGFic29yYmVkIHNpbGVudGx5IGludG8gdGhlCnJlc2lkdWFsIGZhbWlseS4gUm93cyB3aG9zZSBsYWJlbCBpcyBudWxsIGFyZSBkcm9wcGVkIGFuZCBjb3VudGVkLgoKIyMgQTEwLjMgU3Vic2FtcGxpbmcgYW5kIHRoZSByZXN1bHRpbmcgcHJpb3IKClRoZSByZWxlYXNlIGlzIDQ2LjcgbWlsbGlvbiByb3dzLCBiZXlvbmQgdGhlIGNvbXB1dGUgYnVkZ2V0LCBhbmQgc2VjdGlvbiA1IGNhcHMgdHJhaW5pbmcgYnkKc3RyYXRpZmllZCBzdWJzYW1wbGluZy4gKipEZWNpc2lvbjoqKiBzdWJzYW1wbGUgYnkgYSBwZXItbGFiZWwgY2FwIG9mIDYwLDAwMCByb3dzLCBhcHBsaWVkIGFzCmEga2VlcC1mcmFjdGlvbiB3aGlsZSBzdHJlYW1pbmcgZWFjaCBmaWxlLCB3aGljaCByZXRhaW5zIGV2ZXJ5IHJhcmUgbGFiZWwgaW4gZnVsbC4gVGhlIHdvcmtpbmcKZnJhbWUgaXMgMSw1MTAsMTQyIHJvd3MsIDMuMjMgcGVyIGNlbnQgb2YgdGhlIHJlbGVhc2UuCgpUaGlzIGlzIGEgZGVwYXJ0dXJlIGZyb20gcHJvcG9ydGlvbmFsIHN0cmF0aWZpZWQgc3Vic2FtcGxpbmcgYW5kIGl0IG1vdmVzIHRoZSBjbGFzcyBwcmlvci4gQm90aApwcmlvcnMgYXJlIHJlY29yZGVkIGluIHRoZSBmb2NhbC1jbGFzcyByZWNvcmQuIFRoZSBtYXRlcmlhbCBzaGlmdHM6Cgp8IEZhbWlseSB8IFJhdyBzaGFyZSB8IFdvcmtpbmctZnJhbWUgc2hhcmUgfCBFbnJpY2htZW50IHwKfC0tLXwtLS18LS0tfC0tLXwKfCBERG9TIHwgNzIuNzklIHwgNDMuMTklIHwgMC41OXggfAp8IERvUyB8IDE3LjMzJSB8IDE1LjkwJSB8IDAuOTJ4IHwKfCBSZWNvbiB8IDAuNzYlIHwgMTQuNTQlIHwgMTkuMnggfAp8IFNwb29maW5nIHwgMS4wNCUgfCA3LjkzJSB8IDcuNnggfAp8IFdlYiAoZm9jYWwpIHwgMC4wNTMlIHwgMS42NDQlIHwgMzAuOXggfAp8IEJydXRlRm9yY2UgfCAwLjAyOCUgfCAwLjg2NSUgfCAzMC45eCB8CgoqKlJhdGlvbmFsZS4qKiBQcm9wb3J0aW9uYWwgc3Vic2FtcGxpbmcgdG8gYSB0cmFjdGFibGUgc2l6ZSB3b3VsZCBsZWF2ZSB0aGUgcmFyZSBmYW1pbGllcyB3aXRoCnRvbyBmZXcgcG9pbnRzIHRvIGNhbGlicmF0ZSBjbGFzcy1jb25kaXRpb25hbGx5LCB3aGljaCB3b3VsZCBtYWtlIHRoZSBzdHVkeSB1bmFibGUgdG8gbWVhc3VyZQp0aGUgcXVhbnRpdHkgaXQgZXhpc3RzIHRvIG1lYXN1cmUuIFRoZSBjYXAgYWxzbyBtb2RlcmF0ZXMgYSA3MyBwZXIgY2VudCBERG9TIHNoYXJlIHRoYXQgaXMgYW4KYXJ0ZWZhY3Qgb2YgaG93IHRoZSB0ZXN0YmVkIHdhcyBkcml2ZW4gcmF0aGVyIHRoYW4gYW4gb3BlcmF0aW9uYWwgcHJpb3IuCgoqKkRpcmVjdGlvbiBvZiB0aGUgcmVzdWx0aW5nIGJpYXMsIHN0YXRlZCBmb3IgdGhlIHBhcGVyLioqIFRoZSBmb2NhbCBjbGFzcyBpcyBlbnJpY2hlZCByb3VnaGx5CnRoaXJ0eS1mb2xkIHJlbGF0aXZlIHRvIHRoZSByYXcgY2FwdHVyZSwgc28gaXQgaXMgYmV0dGVyIHJlcHJlc2VudGVkIGluIHRyYWluaW5nIHRoYW4gaXQgd291bGQKYmUgaW4gZGVwbG95bWVudCBhbmQgdGhlIGNsYXNzaWZpZXIgc2hvdWxkIGZpbmQgaXQgZWFzaWVyLiBBbnkgZm9jYWwgY292ZXJhZ2UgZmFpbHVyZSBtZWFzdXJlZAppbiB0aGlzIGVudmlyb25tZW50IGlzIHRoZXJlZm9yZSBjb25zZXJ2YXRpdmUgcmF0aGVyIHRoYW4gZXhhZ2dlcmF0ZWQuIE1hcmdpbmFsIGNvdmVyYWdlIGFuZApgU19sYWJgIGFyZSBhZmZlY3RlZCBieSB0aGUgcmV3ZWlnaHRpbmcgYW5kIGFyZSBpbnRlcnByZXRlZCBhZ2FpbnN0IHRoZSByZWNvcmRlZCB3b3JraW5nLWZyYW1lCnByaW9yLCBuZXZlciBhZ2FpbnN0IHRoZSByYXcgY2FwdHVyZSBwcmlvci4KCiMjIEExMC40IEZvY2FsIGNsYXNzCgpTZWN0aW9uIDEyJ3MgcnVsZSwgd2l0aCBvbmUgYWRkZWQgY29uc3RyYWludCBjYXJyaWVkIGZyb20gdGhlIENJQy1JRFMyMDE3IGV4cGVyaWVuY2UKKEFtZW5kbWVudCA5KTogdGhlIGZvY2FsIGNsYXNzIGlzIHRoZSAqKnJhcmVzdCBhdHRhY2sgZmFtaWx5IHRoYXQgc2F0aXNmaWVzIHNlY3Rpb24gNy42IGluIHRoZQpzb3VyY2UgY2FsaWJyYXRpb24gcG9vbCBhbmQgaG9sZHMgYXQgbGVhc3QgdHdvIHN1YnR5cGVzKiouIFRoZSBzZWNvbmQgY29uZGl0aW9uIGlzIG5lY2Vzc2FyeQpiZWNhdXNlIGEgZmFtaWx5IGhlbGQgb3V0IG9mIHRoZSBzb3VyY2UgZW50aXJlbHkgcmV0dXJucyBhbiBpbmZpbml0ZSBxdWFudGlsZSB1bmRlciBTSEMgYW5kIGlzCmV4Y2x1ZGVkIGJ5IHNlY3Rpb24gNy42LCBzbyBpdCBjYW5ub3Qgc2VydmUgYXMgdGhlIGZvY2FsIGNsYXNzIG9mIGEgc2hpZnQgZXhwZXJpbWVudC4KCkFwcGx5aW5nIHRoZSBydWxlOiAqKnRoZSBmb2NhbCBjbGFzcyBpcyBXZWIqKiwgd2l0aCAzLDcyMCBzb3VyY2UgY2FsaWJyYXRpb24gcG9pbnRzIGFnYWluc3QgYQpmbG9vciBvZiAxOSwgYW5kIHNpeCBzdWJ0eXBlcyAoQmFja2Rvb3JfTWFsd2FyZSwgQnJvd3NlckhpamFja2luZywgQ29tbWFuZEluamVjdGlvbiwKU3FsSW5qZWN0aW9uLCBVcGxvYWRpbmdfQXR0YWNrLCBYU1MpLiBCcnV0ZUZvcmNlIGlzIHJhcmVyIGF0IDEsOTU5IHNvdXJjZSBwb2ludHMgYnV0IGhvbGRzIGEKc2luZ2xlIHN1YnR5cGUgYW5kIGlzIHRoZXJlZm9yZSBub3Qgc2hpZnRhYmxlOyB0aGlzIGlzIHJlY29yZGVkIHNvIHRoZSBjaG9pY2UgZG9lcyBub3QgcmVhZCBhcwphbiBvdmVyc2lnaHQuIE5vIGZhbWlseSBmYWlsZWQgZmVhc2liaWxpdHkuCgojIyBBMTAuNSBMYWRkZXIKClNlY3Rpb24gMTAuMSBzcGVjaWZpZXMgZmFtaWx5LWhvbGRvdXQgY29tYmluYXRpb25zIHdpdGggYFNfY292YCBhbmQgYFNfc3VwYCBhcyB0aGUgYW5hbHlzaXMKdmFyaWFibGVzIHJhdGhlciB0aGFuIHRoZSBjb21iaW5hdGlvbiBpbmRleCwgUiA9IDUuIFRoYXQgaXMgcmV0YWluZWQgZm9yIHRoZSAqKnN1cHBvcnQtc2hpZnQKYXJtKiosIGluIHdoaWNoIHdob2xlIGZhbWlsaWVzIGFyZSB3aXRoaGVsZCBmcm9tIHRoZSBzb3VyY2UuCgpUaGUgKipmb2NhbC1jbGFzcyBsYWRkZXIqKiBpcyBhIHZhcmlhbnQgaG9sZG91dCB3aXRoaW4gdGhlIGZvY2FsIGZhbWlseSwgbWlycm9yaW5nIHRoZSBOU0wtS0RECnVuc2Vlbi1zdWJ0eXBlIGxhZGRlciBhbmQgdGhlIENJQy1JRFMyMDE3IHdpdGhpbi1kYXkgRG9TIHZhcmlhbnQgaG9sZG91dDogYSBtb25vdG9uaWNhbGx5CmluY3JlYXNpbmcgZnJhY3Rpb24gb2YgdGhlIGZvY2FsIGNsYXNzJ3MgZXZhbHVhdGlvbiBtYXNzIGlzIGRyYXduIGZyb20gc3VidHlwZXMgd2l0aGhlbGQgZnJvbQpjYWxpYnJhdGlvbiwgd2l0aCB0aGUgZmFtaWx5IGl0c2VsZiBwcmVzZW50IGluIGJvdGggc291cmNlIGFuZCB0YXJnZXQgc28gaXQgcmVtYWlucyBmZWFzaWJsZQp1bmRlciBTSEMuIFIgPSA1IHJlYWxpemF0aW9ucywgcmFuZG9taXplZCBzdWJ0eXBlIHNlbGVjdGlvbiwgbmV2ZXIgb3JkZXJlZCBieSBmcmVxdWVuY3kuCgojIyBBMTAuNiBXaGF0IGlzIHVuY2hhbmdlZAoKUGFydGl0aW9ucyAoc2VjdGlvbiA0LCA2MC8xMC8xNS8xNSBzdHJhdGlmaWVkIGJ5IHNwZWNpZmljIGxhYmVsLCBzZWVkIDIwMjYwNzI2KSwgYmFzZSBtb2RlbHMKYW5kIHNlZWRzICg1KSwgcHJvYmFiaWxpdHkgY2FsaWJyYXRpb24gKDYpLCBjb25mb3JtYWwgc3BlY2lmaWNhdGlvbiBhbmQgZmVhc2liaWxpdHkgKDcpLAptYXRjaGluZyAoOC4xKSwgc2hpZnQgbWVhc3VyZW1lbnQgKDkpLCBhcm1zICgxMC4yKSwgdGhlIGFuYWx5c2lzIHVuaXQgYW5kIG1vZGVsICgxMSksIHRoZQpmb2NhbC1jbGFzcyBydWxlICgxMikgYXMgZXh0ZW5kZWQgaW4gQTEwLjQsIGFuZCBhbGwgc3VjY2VzcyBjcml0ZXJpYSAoMTMpIGFwcGx5IHdpdGhvdXQKbW9kaWZpY2F0aW9uLgoKIyMgQTEwLjcgRmVhdHVyZSBoYW5kbGluZwoKRm9ydHktc2l4IG51bWVyaWMgY29sdW1ucyBhcmUgcmV0YWluZWQgYWZ0ZXIgZXhjbHVkaW5nIGlkZW50aWZpZXItbGlrZSBjb2x1bW5zOyB0d28gY29uc3RhbnQKY29sdW1ucyAoYFRlbG5ldGAsIGBTTVRQYCkgYXJlIGRyb3BwZWQsIGxlYXZpbmcgKio0NCBmZWF0dXJlcyoqLiBObyBpZGVudGlmaWVyLCBhZGRyZXNzIG9yCnRpbWVzdGFtcCBjb2x1bW4gZW50ZXJzIHRoZSBmZWF0dXJlIHNldC4K").decode())
print('wrote preregistration_amendment_10.md')

rec = Path('reports/focal_class_record_ciciot2023.json')
r = json.loads(rec.read_text())
r['version_of_record_note'] = (
    'Full CIC-IoT-2023 CSV release via Kaggle mirror madhavmalhotra/unb-cic-iot-dataset: '
    '169 files, 46,686,579 rows, 34 labels, matching the published release. An earlier '
    'acquisition attempt terminated early at 162 files / 44,025,666 rows and was NOT used '
    'for any analysis; see Amendment 10 A10.1.')
r['release_files'] = 169
r['release_rows'] = 46686579
r['release_labels'] = 34
r['working_frame_rows'] = 1510142
r['amendment'] = 'preregistration_amendment_10.md'
rec.write_text(json.dumps(r, indent=2, default=str))
print('corrected version-of-record note in', rec.name)

dev = Path('reports/deviations.md')
dev.write_text(dev.read_text().rstrip() + """

## nb31 (CIC-IoT-2023) - subsampling
Per-label cap of 60,000 rows applied as a streaming keep-fraction, rather than proportional
stratified subsampling to a row budget (section 5). Retains every rare label in full; the
working frame is 1,510,142 rows, 3.23 per cent of the release. This moves the class prior:
the focal class Web goes from 0.053 per cent of the raw capture to 1.644 per cent of the
working frame, an enrichment of about 31x. Both priors are recorded in the focal-class record.
The enrichment makes the focal class easier for the classifier than it would be in deployment,
so any focal coverage failure measured here is conservative. Recorded before any coverage.

## nb31 (CIC-IoT-2023) - focal-class rule extension
Section 12 selects the rarest attack family satisfying section 7.6. Extended to require at
least two subtypes, because a single-subtype family cannot be shifted by variant holdout while
remaining feasible under SHC (the CIC-IDS2017 lesson, Amendment 9). BruteForce is rarer
(1,959 source calibration points) but single-subtype, so the focal class is Web (3,720 points,
six subtypes). Recorded before any coverage. See Amendment 10 A10.4.

## Primary analysis reassignment (manuscript-level, applies to all environments)
The preregistered primary test is beta5, the SHC x S_cov interaction in the pooled
cross-dataset model (section 11.2, section 13.1). That interaction is not identified at this
design: S_cov varies almost entirely between dataset clusters and the coefficient is
sign-unstable across estimators. The manuscript therefore reports the pooled model as
secondary and descriptive, and rests its causal claim on the within-dataset dose-response.
This is a departure from section 11 and is disclosed as such in the paper.
""")
print('appended 3 entries to deviations.md')

def git(*a):
    r = subprocess.run(['git', *a], capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()); return r
git('add', 'preregistration_amendment_10.md', 'reports/focal_class_record_ciciot2023.json',
    'reports/deviations.md')
git('commit', '-m', 'amendment 10: CIC-IoT-2023 taxonomy, subsampling prior, focal-class rule extension; correct version-of-record; log primary-analysis reassignment')
git('push')
git('log', '--oneline', '-3')

wrote preregistration_amendment_10.md
corrected version-of-record note in focal_class_record_ciciot2023.json
appended 3 entries to deviations.md

[main b8340ed] amendment 10: CIC-IoT-2023 taxonomy, subsampling prior, focal-class rule extension; correct version-of-record; log primary-analysis reassignment
 3 files changed, 145 insertions(+), 2 deletions(-)
 create mode 100644 preregistration_amendment_10.md
To https://github.com/anasbiswas1/calshift-research.git
   88a805c..b8340ed  main -> main
b8340ed amendment 10: CIC-IoT-2023 taxonomy, subsampling prior, focal-class rule extension; correct version-of-record; log primary-analysis reassignment
88a805c nb31: CIC-IoT-2023 acquire, thin, partition, feasibility and focal-class record (before any coverage)
0372432 nb30a: CIC-IoT-2023 acquisition manifest and label inventory (no coverage computed)


CompletedProcess(args=['git', 'log', '--oneline', '-3'], returncode=0, stdout='b8340ed amendment 10: CIC-IoT-2023 taxonomy, subsampling prior, focal-class rule extension; correct version-of-record; log primary-analysis reassignment\n88a805c nb31: CIC-IoT-2023 acquire, thin, partition, feasibility and focal-class record (before any coverage)\n0372432 nb30a: CIC-IoT-2023 acquisition manifest and label inventory (no coverage computed)\n', stderr='')